# Preprocesamiento: Escalado de Características

## Pasos previos

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.model_selection import train_test_splithousing = pd.read_csv("./data/housing.csv")train_set, test_set = train_test_split(housing, test_size=0.2,    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]),    random_state=42    )X_train = train_set.drop("median_house_value", axis=1)y_train = train_set["median_house_value"].copy()X_train_num = X_train.select_dtypes(include=[np.number])

## Escalado, normalización y estandarización

La mayoría de los algoritmos de Machine Learning no funcionan bien cuando las características tienen escalas muy diferentes. Por ejemplo, muchos clasificadores calculan la distancia entre dos puntos usando la distancia euclídea. Si una de las características tiene valores mucho mayores que las demás, la distancia estará dominada por esa característica. Por ejemplo, en nuestro conjunto de datos, el rango de `median_income` va de 0 a 15, mientras que el rango de `total_rooms` va de 6 a 39.320.

Para evitar esto, es habitual escalar las características.

La terminología puede resultar confusa en este punto. En general, la **normalización** se refiere a cambiar la escala de los datos para que se ajusten a un rango específico, mientras que la **estandarización** se refiere a cambiar la distribución de los datos para que tenga una media de 0 y una desviación estándar de 1. En ambos casos son transformaciones lineales que no cambian la forma de la distribución de los datos. En estadística suele haber una distinción clara entre ambos términos, pero en aprendizaje profundo y visión por computador la terminología puede ser menos consistente y **es habitual usar "normalización" para referirse a la estandarización**.

In [ ]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
longitude,16512.0,-119.575635,2.001828,-124.3500,-121.80000,-118.51000,-118.010000,-114.3100
latitude,16512.0,35.639314,2.137963,32.5400,33.94000,34.26000,37.720000,41.9500
housing_median_age,16512.0,28.653404,12.574819,1.0000,18.00000,29.00000,37.000000,52.0000
total_rooms,16512.0,2622.539789,2138.417080,6.0000,1443.00000,2119.00000,3141.000000,39320.0000
total_bedrooms,16354.0,534.914639,412.665649,2.0000,295.00000,433.00000,644.000000,6210.0000
population,16512.0,1419.687379,1115.663036,3.0000,784.00000,1164.00000,1719.000000,35682.0000
households,16512.0,497.011810,375.696156,2.0000,279.00000,408.00000,602.000000,5358.0000
median_income,16512.0,3.875884,1.904931,0.4999,2.56695,3.54155,4.745325,15.0001


## MinMaxScaler

La normalización más habitual es la **normalización min-max** o **escalado min-max**. La **normalización min-max** es la más sencilla: los valores se escalan y desplazan para que queden en el rango entre un valor mínimo y uno máximo. Normalmente será entre 0 y 1, aunque pueden ser otros (las Redes Neuronales suelen funcionar mejor con entradas de media 0, por lo que a veces se usa el rango -1 a 1). Scikit-Learn proporciona una clase `MinMaxScaler` para esto.

$$ X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}} $$

In [ ]:
from sklearn.preprocessing import MinMaxScalermin_max_scaler = MinMaxScaler(feature_range=(-1, 1))housing_num_min_max_scaled = min_max_scaler.fit_transform(X_train_num)

La normalización min-max es muy sensible a los *valores atípicos*, ya que un único valor muy grande puede cambiar completamente la escala de los datos. En una situación en que todos los datos estén entre 20 y 30 pero aparezca un único valor de 100, el máximo pasa a ser 100, desplazando todos los demás valores a un rango muy bajo. En general, la normalización min-max solo debe usarse si estamos seguros de que los *valores atípicos* no son errores. Se pueden aplicar previamente técnicas como el recorte (*capping*) o la Winsorización para limitar el impacto de los valores extremos (consulta [valores atípicos y valores acotados](e2e020_eda.ipynb#outliers-and-capped-values)).

## StandardScaler

Por otro lado, la **estandarización Z-score** (***standard score***) es diferente: primero resta la media (con lo que queda en 0) y luego divide por la **desviación estándar** para que la distribución resultante tenga desviación estándar 1. A diferencia del escalado min-max, la estandarización no limita los valores a un rango específico, pero esto también tiene la ventaja de ser mucho menos sensible a los valores atípicos. Scikit-Learn proporciona una clase `StandardScaler` para esto.

$$ X_{std} = \frac{X - \mu}{\sigma} $$

In [ ]:
from sklearn.preprocessing import StandardScalerstd_scaler = StandardScaler()housing_num_std_scaled = std_scaler.fit_transform(X_train_num)

Muchos modelos de ML funcionan mejor estandarizando las características de entrada y es una práctica común y sistemática en la mayoría de los casos (salvo en modelos basados en árboles). Escalar el objetivo es menos común, pero puede ser útil en algunos casos, especialmente para modelos basados en gradientes (como las Redes Neuronales) o modelos basados en distancias (como KNN o regresiones SVM).

Por ejemplo, podríamos aplicar `StandardScaler` de nuevo a las etiquetas:

In [ ]:
target_scaler = StandardScaler()
scaled_labels = target_scaler.fit_transform(y_train.to_frame()) # convertir el objetivo a dataframe (fit_transform espera 2D)
print(type(y_train)) # Como es una sola columna, las etiquetas estaban almacenadas en un objeto serie
scaled_labels

## Escalado de variables objetivo e inversión posterior

Si transformamos la variable objetivo de alguna forma, la salida de nuestro modelo también devolverá predicciones transformadas. Si queremos que las predicciones estén en la escala original, necesitaremos invertir la transformación. Muchos transformadores de Scikit-Learn tienen un método `inverse_transform()`, que facilita calcular la inversa de sus transformaciones.

> **Nota:** No todas las transformaciones son invertibles. Por ejemplo:
> - **Escaladores numéricos** (`StandardScaler`, `MinMaxScaler`): Siempre invertibles; almacenan los parámetros necesarios para revertir la transformación.
> - **`OneHotEncoder`**: Técnicamente invertible mediante `inverse_transform()`, pero se pierde información si no se conocen las categorías originales.
> - **`OrdinalEncoder`**: Invertible si se conservan las categorías.
> - **Transformaciones con pérdida de información**: Algunas transformaciones como la discretización basada en cuantiles o el redondeo agresivo no pueden invertirse perfectamente.

Para dar un ejemplo simplificado, vamos a entrenar una **regresión lineal simple con el predictor más correlacionado** (`median_income`) y las etiquetas que acabamos de escalar. Luego probaremos sus predicciones con el conjunto de prueba y desharemos la transformación.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train[["median_income"]], scaled_labels) # entrenar el modelo con variables independientes escaladas
some_new_data = X_train[["median_income"]].iloc[:5]  # por simplicidad, simulamos nuevas entradas para predecir tomando 5 filas (no hemos preprocesado el conjunto de prueba)
scaled_predictions = model.predict(some_new_data)
target_scaler.inverse_transform(scaled_predictions) # Deshacer la transformación para obtener predicciones en la escala original

Este proceso se puede simplificar usando la clase `TransformedTargetRegressor` de Scikit-Learn, que permite entrenar un modelo con etiquetas transformadas y deshacer la transformación automáticamente.

In [ ]:
from sklearn.compose import TransformedTargetRegressor
model = TransformedTargetRegressor(regressor = LinearRegression(),
                                   transformer = StandardScaler()) # transformador de la variable dependiente
model.fit(X_train[["median_income"]], y_train)
model.predict(some_new_data)

## Distribuciones de cola pesada

Las **distribuciones de cola pesada** (también llamadas distribuciones sesgadas a la derecha o con asimetría positiva) tienen una cola larga que se extiende hacia valores grandes. Esto significa que unos pocos puntos de datos toman valores extremadamente grandes en comparación con la mayoría. Ejemplos comunes incluyen los ingresos, los recuentos de población y los precios de las viviendas.

> **Nota sobre la asimetría**: Aunque las distribuciones pueden estar sesgadas hacia la izquierda (asimetría negativa) o hacia la derecha (asimetría positiva), **las distribuciones sesgadas a la derecha son mucho más comunes en conjuntos de datos reales**. Esto se debe a que muchas variables (como precios, recuentos o medidas físicas) tienen un límite inferior natural de cero pero no tienen límite superior, lo que permite valores extremos solo en la dirección positiva. Las distribuciones sesgadas a la izquierda (por ejemplo, la edad al morir, las puntuaciones en exámenes) son menos frecuentes en este contexto y suelen manejarse de forma diferente (por ejemplo, mediante transformación cuadrática o exponencial).

### Por qué las colas pesadas son problemáticas para los modelos de ML

1. **Problemas de optimización basada en gradientes**: Los modelos entrenados con descenso por gradiente (regresión lineal, regresión logística, Redes Neuronales) calculan los gradientes en función de la magnitud de los valores de las características. Los valores extremos producen gradientes desproporcionadamente grandes, provocando actualizaciones inestables y una convergencia lenta o errática.
2. **Distorsión basada en distancias**: Los algoritmos que dependen de métricas de distancia (KNN, K-Means, SVM con kernel RBF) se ven gravemente afectados. Una sola característica con valores extremos puede dominar el cálculo de la distancia, haciendo que las demás características sean prácticamente irrelevantes.
3. **Inestabilidad de coeficientes**: En modelos lineales, las características con rangos grandes dan lugar a coeficientes muy pequeños que son numéricamente inestables y más difíciles de interpretar.
4. **Sensibilidad a valores atípicos**: Las colas pesadas contienen inherentemente valores que se comportan como valores atípicos, desplazando los límites de decisión o las líneas de regresión lejos de la masa de datos.

### ¿Qué modelos se ven más afectados?

| Tipo de modelo | Sensibilidad | Razón |
|------------|-------------|--------|
| **Regresión lineal/logística** | Alta | Magnitud del gradiente, escala de coeficientes |
| **Redes Neuronales** | Alta | Entrenamiento basado en gradientes, saturación de activaciones |
| **KNN, K-Means** | Alta | Cálculos de distancia dominados por valores grandes |
| **SVM** | Alta | Cómputos de kernel basados en distancias |
| **Basados en árboles (RandomForest, XGBoost)** | Baja | Las particiones se basan en rangos/umbrales, no en magnitudes |
| **Naive Bayes** | Moderada | Depende de los supuestos de distribución |

### La transformación logarítmica

La transformación logarítmica comprime la cola larga al mapear los valores grandes más cerca unos de otros mientras separa los valores pequeños. Esto hace que la distribución sea más simétrica (más cercana a la normal), lo cual:

- Reduce la influencia de los valores extremos
- Estabiliza la varianza a lo largo del rango
- Mejora el comportamiento del gradiente durante la optimización
- Hace que los cálculos de distancia sean más significativos

**Importante**: Después de aplicar el logaritmo, se recomienda el escalado estándar para garantizar que todas las características estén en una escala comparable.

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

# Distribución original
X_train["population"].hist(ax=axs[0, 0], bins=50)
axs[0, 0].set_xlabel("Población")
axs[0, 0].set_ylabel("Número de distritos")

# Transformación logarítmica
log_pop = X_train["population"].apply(np.log)
log_pop.hist(ax=axs[0, 1], bins=50)
axs[0, 1].set_xlabel("Logaritmo de la población")
axs[0, 1].set_ylabel("Número de distritos")

# Escalado estándar
scaler = StandardScaler()
scaled_pop = scaler.fit_transform(X_train["population"].values.reshape(-1, 1)).flatten()
axs[1, 0].hist(scaled_pop, bins=50)
axs[1, 0].set_xlabel("Población escalada")
axs[1, 0].set_ylabel("Número de distritos")

# Transformación logarítmica + escalado estándar
scaled_log_pop = scaler.fit_transform(log_pop.values.reshape(-1, 1)).flatten()
axs[1, 1].hist(scaled_log_pop, bins=50)
axs[1, 1].set_xlabel("Logaritmo de la población escalado")
axs[1, 1].set_ylabel("Número de distritos")

plt.tight_layout()
plt.show()